<a href="https://colab.research.google.com/github/Khaledsaads/gemma-7b-sft-dpo/blob/main/gemma_7b_dpo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## install needed libraries



In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -- no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-oe_5_9gf/unsloth_a4fd93eb513048988881af315989d062
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-oe_5_9gf/unsloth_a4fd93eb513048988881af315989d062
  Resolved https://github.com/unslothai/unsloth.git to commit 40840d7212838b4c67e21f821532614c7a7f65b1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 28.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 56.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 79.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 58.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 140.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 142.9 MB/s  0:00:00
  Created wheel for unsl

In [2]:
! pip install trl

In [3]:
!pip install -U torchao

In [4]:
! pip install transformers

In [5]:
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

In [6]:
from trl import DPOTrainer, DPOConfig
from peft import PeftModel
from transformers import AutoTokenizer
from datasets import load_dataset
from unsloth import FastLanguageModel

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1689: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Load and Explore dataset

In [7]:
dataset = load_dataset("argilla/dpo-mix-7k")

In [8]:
from unsloth.chat_templates import get_chat_template
tokenizer = AutoTokenizer.from_pretrained("Khaledsaad1/gemma-7b-deita-10k-qlora")
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma")

In [9]:
for key in dataset["train"][0].keys():
  print(f"{key}: {dataset["train"][0][key]}")

dataset: argilla/distilabel-intel-orca-dpo-pairs
chosen: [{'content': 'Q:Question: how old julio cesar chavez when he fought de la hoya I found the following answer on Google: He holds records for most successful consecutive defenses of world titles (27), most title fights (37), most title-fight victories (31) and he is after Joe Louis with (23) for most title defenses won by knockout (21). Is that a correct answer? Yes or no.\nA:', 'role': 'user'}, {'content': " Yes, the information you found on Google is correct. Julio César Chávez holds several records related to world title defenses and victories, and he is considered one of the greatest boxers in history. Here is a detailed answer to your question:\n\nJulio César Chávez was born on July 12, 1962, in Ciudad Obregón, Sonora, Mexico. He began boxing at a young age and quickly made a name for himself in the sport, winning his first world title in 1984 when he defeated Mario Miranda for the WBC super featherweight championship.\n\nOver

In [10]:
def apply_dpo_template(example):
  if all(k in example.keys() for k in ("chosen", "rejected")):
    prompt_text = example["chosen"][:-1]
    chosen_text = example["chosen"][-1:]
    rejected_text = example["rejected"][-1:]
    example["text_prompt"] = tokenizer.apply_chat_template(prompt_text, tokenize= False)
    example["text_chosen"] = tokenizer.apply_chat_template(chosen_text, tokenize= False)
    example["text_rejected"] = tokenizer.apply_chat_template(rejected_text, tokenize= False)
  return example

In [11]:
dataset["train"]

Dataset({
    features: ['dataset', 'chosen', 'rejected', 'chosen_rating', 'rejected_rating'],
    num_rows: 6750
})

In [12]:
dataset['test']

Dataset({
    features: ['dataset', 'chosen', 'rejected', 'chosen_rating', 'rejected_rating'],
    num_rows: 750
})

In [13]:
columns_names = (dataset['train'].features)
dataset_template = dataset.map(apply_dpo_template, remove_columns = columns_names,desc="Formatting comparisons with prompt template")
for split in ["train", "test"]:
  dataset_template[split] = dataset_template[split].rename_columns(
      {"text_prompt": "prompt", "text_chosen":"chosen", "text_rejected": "rejected"}
  )

In [14]:
dataset_template["train"].features

{'prompt': Value('string'),
 'chosen': Value('string'),
 'rejected': Value('string')}

In [15]:
dataset_template["test"].features

{'prompt': Value('string'),
 'chosen': Value('string'),
 'rejected': Value('string')}

In [16]:
dataset_template

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 6750
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 750
    })
})

In [17]:
for key in dataset_template["train"][0].keys():
  print(f"{key}: {dataset_template["train"][0][key]}")

prompt: <bos><start_of_turn>user
Q:Question: how old julio cesar chavez when he fought de la hoya I found the following answer on Google: He holds records for most successful consecutive defenses of world titles (27), most title fights (37), most title-fight victories (31) and he is after Joe Louis with (23) for most title defenses won by knockout (21). Is that a correct answer? Yes or no.
A:<end_of_turn>

chosen: <bos><start_of_turn>model
Yes, the information you found on Google is correct. Julio César Chávez holds several records related to world title defenses and victories, and he is considered one of the greatest boxers in history. Here is a detailed answer to your question:

Julio César Chávez was born on July 12, 1962, in Ciudad Obregón, Sonora, Mexico. He began boxing at a young age and quickly made a name for himself in the sport, winning his first world title in 1984 when he defeated Mario Miranda for the WBC super featherweight championship.

Over the course of his career, C

In [18]:
import numpy as np
import json
for feature in dataset_template["train"].features:
  lengths = [len(tokenizer(json.dumps(text))["input_ids"]) for text in dataset_template['train'][feature]]
  print(f"{feature} p99: {np.percentile(lengths, 99)}")
  print(f"{feature} p95: {np.percentile(lengths, 95)}")
  print(f"{feature} p90: {np.percentile(lengths, 90)}")

prompt p99: 2161.040000000001
prompt p95: 1451.0
prompt p90: 1112.0
chosen p99: 1017.590000000002
chosen p95: 759.0999999999985
chosen p90: 621.1000000000004
rejected p99: 1192.320000000007
rejected p95: 802.0
rejected p90: 640.0


## Load Model and Tokenizer

In [19]:
model , tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Khaledsaad1/gemma-7b-deita-10k-qlora",
    dtype =None,
    max_seq_length = 2048,  # p90 of max(prompt+ projected, prompet+chosen) = 1752
    use_gradient_checkpointing=False,
    load_in_4bit = True
)

==((====))==  Unsloth 2026.10.8: Fast Gemma patching. Transformers: 5.17.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-7b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.10.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [20]:
model.print_trainable_parameters()

trainable params: 50,003,968 || all params: 8,587,684,864 || trainable%: 0.5823


In [22]:
# use gemma template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma")

Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer__cu9657_/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/_unsloth_sentencepiece_temp/tokenizer__cu9657_.


In [23]:
from unsloth import FastLanguageModel
from unsloth import PatchDPOTrainer
PatchDPOTrainer()
from trl import DPOTrainer, DPOConfig
from transformers import EarlyStoppingCallback

## Initialize trainer arguments

In [24]:
trainer_args = DPOConfig(
    output_dir="./gemma_7B_dpo_model",
    num_train_epochs = 2,
    per_device_train_batch_size =1,
    per_device_eval_batch_size =16,
    gradient_accumulation_steps=16,
    eval_strategy = "steps",
    save_strategy = "steps",
    save_steps = 80,
    eval_steps = 80,
    logging_steps =40,
    warmup_steps = 40,
    beta = 0.05,
    learning_rate = 2e-5,
    optim = "paged_adamw_8bit",
    lr_scheduler_type = "cosine",
    weight_decay = 0.05,
    gradient_checkpointing =False,
    bf16=True,
    fp16= False,
    max_length = 2048,
    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
    greater_is_better  = False,
    torch_compile=False
)

## Initialize trainer

In [25]:
from transformers import EarlyStoppingCallback
trainer = DPOTrainer(
    model,
    ref_model = None,
    train_dataset = dataset_template["train"],
    eval_dataset = dataset_template["test"],
    args = trainer_args,
    processing_class = tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

Tokenizing train dataset (num_proc=6):   0%|          | 0/6750 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset (num_proc=6):   0%|          | 0/6750 [00:00<?, ? example…

Tokenizing eval dataset (num_proc=6):   0%|          | 0/750 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset (num_proc=6):   0%|          | 0/750 [00:00<?, ? examples/…

## Start Training

In [26]:
trainer.train(resume_from_checkpoint="/content/gemma_7B_dpo_model/checkpoint-480")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,709 | Num Epochs = 2 | Total steps = 840
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 50,003,968 of 8,637,688,832 (0.58% trained)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
560,0.631789,0.582193,0.948169,1619205.000000,210.950698,212.480805,0.668231,0.242348,-1.022412,0.718972,1.264760,-2239.169241,-2299.930233
640,0.502086,0.540391,1.011780,3241797.000000,218.107279,219.566620,0.655136,-2.556625,-5.176104,0.758865,2.619479,-2295.148718,-2383.004088
720,0.571136,0.460994,1.022097,4895448.000000,223.949314,225.381781,0.666038,-0.782890,-2.916063,0.782506,2.133172,-2259.674026,-2337.803264
800,0.480923,0.465995,0.936030,6513373.000000,225.116599,226.717860,0.669123,-0.800746,-2.892558,0.783540,2.091812,-2260.031125,-2337.333158
840,0.491203,0.455060,0.932338,7295367.000000,225.120839,226.755721,0.669100,-0.874771,-2.999195,0.800827,2.124424,-2261.511641,-2339.465893


Unsloth: Restored added_tokens_decoder metadata in ./gemma_7B_dpo_model/checkpoint-560/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./gemma_7B_dpo_model/checkpoint-640/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./gemma_7B_dpo_model/checkpoint-640.
Unsloth: Restored added_tokens_decoder metadata in ./gemma_7B_dpo_model/checkpoint-720/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./gemma_7B_dpo_model/checkpoint-720.
Unsloth: Restored added_tokens_decoder metadata in ./gemma_7B_dpo_model/checkpoint-800/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./gemma_7B_dpo_model/checkpoint-800.
Unsloth: Restored added_tokens_decoder metadata in ./gemma_7B_dpo_model/checkpoint-840/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./gemma_7B_dpo_model/checkpoint-840.


TrainOutput(global_step=840, training_loss=0.24137933821905228, metrics={'train_runtime': 5355.7268, 'train_samples_per_second': 2.505, 'train_steps_per_second': 0.157, 'total_flos': 8.938522117815583e+17, 'train_loss': 0.24137933821905228, 'epoch': 2.0})

## LR during training

In [29]:
path = "/content/gemma_7B_dpo_model/checkpoint-840/trainer_state.json"
with open(path, "r") as f:
    state = json.load(f)

for entry in state["log_history"]:
    if "learning_rate" in entry:
        print(f"Step: {entry.get('step')}, LR: {entry['learning_rate']}")

Step: 40, LR: 0.000195
Step: 80, LR: 0.00019882950400358694
Step: 120, LR: 0.00019522626868413956
Step: 160, LR: 0.0001892782465918221
Step: 200, LR: 0.000181131897770901
Step: 240, LR: 0.00017098781236552903
Step: 280, LR: 0.00015909577142467573
Step: 320, LR: 0.00014574859646372605
Step: 360, LR: 0.0001312749392269526
Step: 400, LR: 0.00011603118919083913
Step: 440, LR: 0.0001003926980723806
Step: 480, LR: 8.474453742408992e-05
Step: 520, LR: 6.947201689439565e-05
Step: 560, LR: 5.495119662532857e-05
Step: 600, LR: 4.1539627403752666e-05
Step: 640, LR: 2.956754657434937e-05
Step: 680, LR: 1.9329746500197023e-05
Step: 720, LR: 1.1078315796431571e-05
Step: 760, LR: 5.0164320719082285e-06
Step: 800, LR: 1.2933590221634428e-06
Step: 840, LR: 7.710618529443991e-10


In [30]:
from transformers import TrainingArguments
import torch
path = "/content/gemma_7B_dpo_model/checkpoint-840/training_args.bin"
args = torch.load(path, map_location="cpu", weights_only=False)
print("Configured LR:", args.learning_rate)

Configured LR: 2e-05


In [31]:
from huggingface_hub import login
login()

## Save best Checkpoint

In [32]:
from huggingface_hub import HfApi
api = HfApi()
repo_id = "Khaledsaad1/argilla-dpo-mix-7k-DPO-840-checkpoint"
api.create_repo(
    repo_id = repo_id,
    repo_type = "model" ,
    exist_ok =True

    )
api.upload_folder(
    repo_id = repo_id,
    repo_type = "model",
    folder_path = "/content/gemma_7B_dpo_model/checkpoint-840")

CommitInfo(commit_url='https://huggingface.co/Khaledsaad1/argilla-dpo-mix-7k-DPO-840-checkpoint/commit/0b634327332c7a1a39c05d4f4e9c12e825cda0cc', commit_message='Upload folder using huggingface_hub', commit_description='', oid='0b634327332c7a1a39c05d4f4e9c12e825cda0cc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Khaledsaad1/argilla-dpo-mix-7k-DPO-840-checkpoint', endpoint='https://huggingface.co', repo_type='model', repo_id='Khaledsaad1/argilla-dpo-mix-7k-DPO-840-checkpoint'), pr_revision=None, pr_num=None)

In [2]:
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Load model with best checkpoint

In [3]:
model , tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Khaledsaad1/argilla-dpo-mix-7k-DPO-840-checkpoint",
    dtype =None,
    max_seq_length = 2048,
    use_gradient_checkpointing=False,
    load_in_4bit = True
)

==((====))==  Unsloth 2026.10.8: Fast Gemma patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-7b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.10.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (GemmaDecoderLayer).


In [5]:
from unsloth.chat_templates import get_chat_template
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Khaledsaad1/gemma-7b-deita-10k-qlora")
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma")

tokenizer_config.json:   0%|          | 0.00/39.8k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/638 [00:00<?, ?B/s]

## Test Model on Unseen data
Note: created by gemini

In [40]:
benchmark_examples= [
  [
    { "role": "user", "content": "Write a Python function to check if a string is a palindrome, ignoring spaces and punctuation." },
    { "role": "assistant", "content": "def is_palindrome(s: str) -> bool:\n    cleaned = ''.join(ch.lower() for ch in s if ch.isalnum())\n    return cleaned == cleaned[::-1]" }
  ],
  [
    { "role": "user", "content": "Calculate the compound interest for a principal of $1,000 at an annual rate of 5% compounded annually for 3 years." },
    { "role": "assistant", "content": "The formula for compound interest is $A = P(1 + r/n)^{nt}$. \nHere, $P = 1000$, $r = 0.05$, $n = 1$, and $t = 3$.\n$A = 1000 \times (1 + 0.05)^3 = 1000 \times 1.157625 = 1157.63$.\nThus, the total amount is $1,157.63, yielding $157.63 in interest." }
  ],
  [
    { "role": "user", "content": "Provide a brief overview of the Gutenberg printing press and its impact on history." },
    { "role": "assistant", "content": "Invented by Johannes Gutenberg around 1440, the printing press introduced movable metal type to Europe. It revolutionized communication by mass-producing books, dramatically lowering costs, increasing literacy rates, and accelerating the spread of scientific and religious knowledge." }
  ],
  [
    { "role": "user", "content": "What is the difference between supervised and unsupervised machine learning?" },
    { "role": "assistant", "content": "Supervised learning uses labeled datasets to train models to predict outcomes or classify data accurately. Unsupervised learning works with unlabeled data, allowing the algorithm to discover hidden patterns, groupings (clustering), or underlying structures on its own." }
  ],
  [
    { "role": "user", "content": "Give me a quick Bash command to find all files larger than 100MB in the current directory." },
    { "role": "assistant", "content": "You can use the `find` command:\n`find . -type f -size +100M`" }
  ],
  [
    { "role": "user", "content": "Write a SQL query to find the top 3 highest-paid employees from an 'employees' table." },
    { "role": "assistant", "content": "SELECT name, salary FROM employees ORDER BY salary DESC LIMIT 3;" }
  ],
  [
    { "role": "user", "content": "Explain the concept of 'opportunity cost' in economics with a simple example." },
    { "role": "assistant", "content": "Opportunity cost is the potential benefit given up when choosing one alternative over another. For example, if you spend $10 on a movie ticket, the opportunity cost is whatever else you could have bought or enjoyed with that $10, like a book or a meal." }
  ],
  [
    { "role": "user", "content": "What are the primary macronutrients required by the human body?" },
    { "role": "assistant", "content": "The three primary macronutrients are carbohydrates (the body's main energy source), proteins (essential for tissue repair and growth), and fats (vital for hormone regulation and energy storage)." }
  ],
  [
    { "role": "user", "content": "Write a short JavaScript snippet to toggle a CSS class on an element when clicked." },
    { "role": "assistant", "content": "const button = document.querySelector('#myButton');\nbutton.addEventListener('click', () => {\n  document.body.classList.toggle('dark-mode');\n});" }
  ],
  [
    { "role": "user", "content": "What causes the phases of the Moon as seen from Earth?" },
    { "role": "assistant", "content": "The phases of the Moon are caused by its changing position relative to the Earth and the Sun as it orbits our planet. This changes the angle at which sunlight hits the lunar surface visible from Earth." }
  ],
  [
    { "role": "user", "content": "Provide a helpful tip for managing time effectively using the Pomodoro Technique." },
    { "role": "assistant", "content": "Work with intense focus for 25 minutes without distractions, then take a mandatory 5-minute break. After completing four cycles, take a longer break of 15 to 30 minutes to maintain sustainable productivity." }
  ],
  [
    { "role": "user", "content": "What is a RESTful API and what are its key HTTP methods?" },
    { "role": "assistant", "content": "A RESTful API is an architectural style for designing networked applications using HTTP requests. Key methods include GET (retrieve data), POST (create data), PUT/PATCH (update data), and DELETE (remove data)." }
  ],
  [
    { "role": "user", "content": "How do you declare a constant variable in modern JavaScript?" },
    { "role": "assistant", "content": "You use the `const` keyword. For example:\n`const API_URL = 'https://api.example.com';`\nNote that `const` prevents re-assignment of the variable identifier, though mutable objects assigned to it can still have their properties modified." }
  ],
  [
    { "role": "user", "content": "Define 'recursion' in computer science simply." },
    { "role": "assistant", "content": "Recursion is a programming technique where a function calls itself to solve smaller sub-instances of the same problem until it reaches a base case that stops the recursion." }
  ],
  [
    { "role": "user", "content": "What is the capital of Australia, and why is it often confused?" },
    { "role": "assistant", "content": "The capital of Australia is Canberra. It is frequently confused with larger coastal cities like Sydney or Melbourne because those are major global and cultural hubs." }
  ],
  [
    { "role": "user", "content": "Write a CSS rule to center a flexbox item both horizontally and vertically." },
    { "role": "assistant", "content": ".container {\n  display: flex;\n  justify-content: center;\n  align-items: center;\n}" }
  ],
  [
    { "role": "user", "content": "What is the difference between a process and a thread in operating systems?" },
    { "role": "assistant", "content": "A process is an independent execution unit with its own memory space and system resources. A thread is a smaller unit of execution within a process, sharing the same memory space to allow for faster context switching and communication." }
  ],
  [
    { "role": "user", "content": "Give me a quick tip on how to structure a professional cover letter." },
    { "role": "assistant", "content": "Keep it to one page. Structure it with an engaging opening stating the role you want, 1-2 body paragraphs highlighting specific achievements matching their needs, and a strong closing call to action expressing enthusiasm." }
  ],
  [
    { "role": "user", "content": "How does photosynthesis convert light energy into chemical energy?" },
    { "role": "assistant", "content": "During photosynthesis, plants capture sunlight using chlorophyll in chloroplasts. They use this solar energy to convert carbon dioxide from the air and water from the soil into glucose (sugar) for food, releasing oxygen as a byproduct." }
  ],
  [
    { "role": "user", "content": "What is Git rebase and how does it differ from a merge?" },
    { "role": "assistant", "content": "Git merge combines branches by creating a new 'merge commit,' preserving the exact historical timeline. Git rebase moves or combines a sequence of commits to a new base commit, resulting in a clean, linear commit history." }
  ],
  [
    { "role": "user", "content": "What is the function of mitochondria in human cells?" },
    { "role": "assistant", "content": "Mitochondria are known as the powerhouses of the cell because they generate most of the chemical energy (ATP) needed to power biochemical reactions via cellular respiration." }
  ],
  [
    { "role": "user", "content": "Write a regular expression to match a standard email address format." },
    { "role": "assistant", "content": "^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\\.[a-zA-Z0-9-.]+$" }
  ],
  [
    { "role": "user", "content": "What is inflation and what typically causes it?" },
    { "role": "assistant", "content": "Inflation is the general increase in prices and fall in the purchasing value of money over time. It is typically caused by demand-pull factors (high consumer demand outpacing supply), cost-push factors (increased production costs), or excessive money supply growth." }
  ],
  [
    { "role": "user", "content": "Explain what an index does in a relational database." },
    { "role": "assistant", "content": "A database index is a data structure (like a B-tree) that improves the speed of data retrieval operations on a table at the cost of additional storage space and slower write performance." }
  ],
  [
    { "role": "user", "content": "What are the core pillars of Object-Oriented Programming (OOP)?" },
    { "role": "assistant", "content": "The four core pillars are Encapsulation (bundling data and methods), Inheritance (reusing code across classes), Polymorphism (using a unified interface for different data types), and Abstraction (hiding complex implementation details)." }
  ],
  [
    { "role": "user", "content": "How can you check open ports on a Linux server?" },
    { "role": "assistant", "content": "You can use the `ss` command or netstat:\n`ss -tuln`\nThis displays all listening TCP and UDP ports numerically." }
  ],
  [
    { "role": "user", "content": "What is the Heisenberg Uncertainty Principle in quantum mechanics?" },
    { "role": "assistant", "content": "It states that you cannot simultaneously know both the precise position and momentum of a particle; the more accurately one is measured, the less precisely the other can be known." }
  ],
  [
    { "role": "user", "content": "Write a Python snippet to read a CSV file using the built-in module." },
    { "role": "assistant", "content": "import csv\n\nwith open('data.csv', mode='r') as file:\n    reader = csv.reader(file)\n    for row in reader:\n        print(row)" }
  ],
  [
    { "role": "user", "content": "What is the difference between symmetric and asymmetric encryption?" },
    { "role": "assistant", "content": "Symmetric encryption uses a single secret key for both encryption and decryption, making it fast but requiring secure key sharing. Asymmetric encryption uses a key pair (a public key for encryption and a private key for decryption), enhancing security for key exchange." }
  ],
  [
    { "role": "user", "content": "Give a quick summary of the Pythagorean theorem." },
    { "role": "assistant", "content": "For any right-angled triangle, the square of the length of the hypotenuse ($c$) equals the sum of the squares of the lengths of the other two sides ($a$ and $b$): $a^2 + b^2 = c^2$." }
  ],
  [
    { "role": "user", "content": "What is a Docker container?" },
    { "role": "assistant", "content": "A Docker container is a lightweight, standalone, executable package of software that includes everything needed to run an application—code, runtime, system tools, libraries, and settings—ensuring consistent execution across environments." }
  ],
  [
    { "role": "user", "content": "What are the primary layers of the OSI model?" },
    { "role": "assistant", "content": "The 7 layers from bottom to top are: Physical, Data Link, Network, Transport, Session, Presentation, and Application (remembered often as 'Please Do Not Throw Sausage Pizza Away')." }
  ],
  [
    { "role": "user", "content": "How do you handle asynchronous operations in modern JavaScript?" },
    { "role": "assistant", "content": "Modern JavaScript handles asynchronous operations using `Promises` and `async/await` syntax, which allows writing asynchronous code that looks and behaves like synchronous code." }
  ],
  [
    { "role": "user", "content": "What is the significance of the pH scale in chemistry?" },
    { "role": "assistant", "content": "The pH scale measures how acidic or basic a watery solution is, ranging from 0 to 14. A value of 7 is neutral, values below 7 indicate acidity, and values above 7 indicate alkalinity (basicity)." }
  ],
  [
    { "role": "user", "content": "Provide a simple definition of cloud computing." },
    { "role": "assistant", "content": "Cloud computing is the delivery of computing services—including servers, storage, databases, networking, and software—over the internet ('the cloud') rather than relying on local hardware." }
  ],
  [
    { "role": "user", "content": "Write a simple function in Python to calculate the factorial of a number iteratively." },
    { "role": "assistant", "content": "def factorial(n):\n    result = 1\n    for i in range(2, n + 1):\n        result *= i\n    return result" }
  ],
  [
    { "role": "user", "content": "What is a deadlock in concurrent computing?" },
    { "role": "assistant", "content": "A deadlock is a situation where two or more competing actions are each waiting for the other to finish, resulting in none of them ever proceeding." }
  ],
  [
    { "role": "user", "content": "What is the difference between `let`, `const`, and `var` in JavaScript?" },
    { "role": "assistant", "content": "`var` is function-scoped and allows redeclaration. `let` is block-scoped and allows reassignment but not redeclaration. `const` is block-scoped, prevents reassignment, and requires immediate initialization." }
  ],
  [
    { "role": "user", "content": "Explain how DNS (Domain Name System) works." },
    { "role": "assistant", "content": "DNS acts as the phonebook of the internet, translating human-friendly domain names (like example.com) into machine-readable IP addresses (like 192.0.2.1) through a hierarchical network of name servers." }
  ],
  [
    { "role": "user", "content": "What is a closure in programming languages like JavaScript?" },
    { "role": "assistant", "content": "A closure is a function bundled together with references to its lexical scope environment, allowing an inner function to access variables from an outer function even after the outer function has finished executing." }
  ],
  [
    { "role": "user", "content": "What are the main goals of database normalization?" },
    { "role": "assistant", "content": "Database normalization aims to minimize data redundancy, eliminate undesirable characteristics like insertion, update, and deletion anomalies, and ensure data dependencies make logical sense." }
  ],
  [
    { "role": "user", "content": "What is the difference between HTTP and HTTPS?" },
    { "role": "assistant", "content": "HTTPS (Hypertext Transfer Protocol Secure) is the secure version of HTTP. It uses SSL/TLS encryption to protect data transmitted between a web browser and a website from eavesdropping or tampering." }
  ],
  [
    { "role": "user", "content": "Write a Python script snippet to fetch JSON data from a public API." },
    { "role": "assistant", "content": "import requests\n\nresponse = requests.get('https://api.github.com')\nif response.status_code == 200:\n    data = response.json()\n    print(data)" }
  ],
  [
    { "role": "user", "content": "What is polymorphism in object-oriented programming?" },
    { "role": "assistant", "content": "Polymorphism allows different classes to share the same method interface while implementing behavior specific to their own class, enabling objects to be treated interchangeably through a common parent type." }
  ],
  [
    { "role": "user", "content": "What is the primary purpose of a firewall in network security?" },
    { "role": "assistant", "content": "A firewall monitors and filters incoming and outgoing network traffic based on predetermined security rules, acting as a barrier between a trusted internal network and untrusted external networks like the internet." }
  ],
  [
    { "role": "user", "content": "Give a quick summary of what Agile software development is." },
    { "role": "assistant", "content": "Agile is an iterative approach to project management and software development that helps teams deliver value to customers faster through small, frequent cycles (sprints) rather than a monolithic sequential release plan." }
  ],
  [
    { "role": "user", "content": "What is a hash function and what are its key properties?" },
    { "role": "assistant", "content": "A hash function maps data of arbitrary size to fixed-size values. Key properties include determinism (same input always yields same output), quick computation, preimage resistance (hard to reverse), and avalanche effect (small input change drastically alters output)." }
  ],
  [
    { "role": "user", "content": "What is the difference between a primary key and a foreign key?" },
    { "role": "assistant", "content": "A primary key uniquely identifies each record in a table and cannot contain null values. A foreign key is a field (or collection of fields) in one table that uniquely references a primary key in another table, establishing a relationship between them." }
  ],
  [
    { "role": "user", "content": "What is an algorithm's Big O notation used for?" },
    { "role": "assistant", "content": "Big O notation describes the limiting behavior or complexity of an algorithm, classifying how its execution time or memory footprint scales relative to the input size ($n$) as $n$ grows large." }
  ],
  [
    { "role": "user", "content": "Briefly explain the MVC (Model-View-Controller) architectural pattern." },
    { "role": "assistant", "content": "MVC divides an application into three interconnected parts: the Model manages data and business logic; the View handles UI display and rendering; and the Controller accepts user input, manipulates the Model, and returns the appropriate View." }
  ]
]

In [41]:
i = 0;
for example in benchmark_examples:
  print(example[0])
  i+=1
  if i> 2:
    break

{'role': 'user', 'content': 'Write a Python function to check if a string is a palindrome, ignoring spaces and punctuation.'}
{'role': 'user', 'content': 'Calculate the compound interest for a principal of $1,000 at an annual rate of 5% compounded annually for 3 years.'}
{'role': 'user', 'content': 'Provide a brief overview of the Gutenberg printing press and its impact on history.'}


In [42]:
import torch
text = [{"role": "user", "content": "write a python code that check word is palindrome or not"}]
input_ids_list = []
for example in benchmark_examples:
  input_ids = tokenizer.apply_chat_template(
      [example[0]],
      tokenize = True,
      add_generation_prompt = True,
      return_tensors= "pt").to("cuda")
  input_ids_list.append(input_ids)


In [43]:
tokenizer.decode(input_ids_list[0][0])

'<bos><start_of_turn>user\nWrite a Python function to check if a string is a palindrome, ignoring spaces and punctuation.<end_of_turn>\n<start_of_turn>model\n'

In [44]:
input_ids.shape[-1]

24

In [57]:
import json
FastLanguageModel.for_inference(model)
from tqdm import tqdm
model.eval()
results = []

for input_ids in tqdm(input_ids_list):
  with torch.inference_mode():
    output = model.generate(
        input_ids = input_ids,
        do_sample=True,
        top_p = 0.9,
        top_k = 50,
        temperature = 0.7,
        max_tokens = None,
        use_cache = True
    )
    response =tokenizer.decode(output[0], skip_special_tokens=True)

  sample = []
  sample.append({
      "role": "user",
      "content": tokenizer.decode(
          output[0, 2:input_ids.shape[-1]- 2],
          skip_special_tokens = True
          )
      })
  sample.append({
      "role": "assistant",
      "content": tokenizer.decode(
          output[0, input_ids.shape[-1]:],
          skip_special_tokens = True
          )
      })
  results.append(sample)
with open("test_cases.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)


100%|██████████| 50/50 [30:35<00:00, 36.71s/it]


In [73]:
for sample in results:
    for message in sample:
        if message["role"] == "user":
            message["content"] = message["content"].removeprefix("user\n")
with open("test_cases.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

In [74]:
print(results[0])

[{'role': 'user', 'content': 'Write a Python function to check if a string is a palindrome, ignoring spaces and punctuation.\n'}, {'role': 'assistant', 'content': 'To write a Python function to check if a string is a palindrome, ignoring spaces and punctuation, you can use the following steps:\n\n1. Remove all spaces and punctuation from the string.\n2. Reverse the modified string.\n3. Compare the original and reversed strings. If they are equal, the string is a palindrome.\n\nHere\'s a sample Python function that implements these steps:\n```python\ndef is_palindrome(string):\n    # Remove all spaces and punctuation from the string\n    modified_string = \'\'.join(c for c in string if c.isalpha())\n\n    # Reverse the modified string\n    reversed_string = modified_string[::-1]\n\n    # Compare the original and reversed strings\n    if modified_string == reversed_string:\n        return True\n    else:\n        return False\n\n# Test the function\nprint(is_palindrome("A man, a plan, a 

gemini rate model answers as 10/10

In [81]:
i = 0
for sample in results:
  for role in sample:
    print(role["role"])
    print(role["content"])
  print("#"*100)
  i+=1
  if i > 15:
    break

user
Write a Python function to check if a string is a palindrome, ignoring spaces and punctuation.

assistant
To write a Python function to check if a string is a palindrome, ignoring spaces and punctuation, you can use the following steps:

1. Remove all spaces and punctuation from the string.
2. Reverse the modified string.
3. Compare the original and reversed strings. If they are equal, the string is a palindrome.

Here's a sample Python function that implements these steps:
```python
def is_palindrome(string):
    # Remove all spaces and punctuation from the string
    modified_string = ''.join(c for c in string if c.isalpha())

    # Reverse the modified string
    reversed_string = modified_string[::-1]

    # Compare the original and reversed strings
    if modified_string == reversed_string:
        return True
    else:
        return False

# Test the function
print(is_palindrome("A man, a plan, a canal: Panama"))   # True
print(is_palindrome("A car, a hearse"))             